# 📊 Paper Evaluations — Kaggle (Step 6/8)
**Generates CIs, McNemar, Calibration, Robustness, and Per-Generator metrics.**

## ⚙️ Setup
1. **Datasets** — Add all 12 datasets (Input → Add Data)
2. **Secret** — Add `HF_TOKEN` secret
3. **Accelerator** — GPU T4 x2
4. **Run All**

**Sequence**: Tiny → Base → Large → Baselines → Ablation → **PaperEvals** → LOGO → Outputs

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 0: Clone Repo + Install Dependencies
# ═══════════════════════════════════════════════════════════
import subprocess, sys, os
from pathlib import Path

REPO_URL  = "https://github.com/MIHMahmudEli/ai-image-detection-research.git"
CLONE_DIR = Path("/kaggle/working/ai-image-detection-research")

if not CLONE_DIR.exists():
    print("Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL, str(CLONE_DIR)], check=True)
else:
    print("Pulling latest...")
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull", "--rebase"], check=False)

os.chdir(str(CLONE_DIR))\nsys.path.insert(0, str(CLONE_DIR / "model"))

subprocess.run([sys.executable, "-m", "pip", "install",
    "huggingface_hub", "open_clip_torch", "scipy", "scikit-learn",
    "-q", "--disable-pip-version-check"], check=False)

print(f"Project root: {CLONE_DIR}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 1: Imports & Environment
# ═══════════════════════════════════════════════════════════
import os, sys, math, json, time, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.notebook import tqdm

sys.path.insert(0, str(Path("/kaggle/working/ai-image-detection-research/model")))
from src.kaggle_utils import KaggleEnv

env = KaggleEnv(project_root_search=True)
PROJECT_ROOT = env.project_root
os.chdir(env.working_dir)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SMOKE_TEST = not torch.cuda.is_available()
print(f"Device: {device} | SMOKE: {SMOKE_TEST}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 2: Dataloaders
# ═══════════════════════════════════════════════════════════
from src.dataset import create_split_dataloaders
from src.config import Config

cfg = Config()
VARIANT = "base"
IMAGE_SIZE  = 224 if SMOKE_TEST else 384
BATCH_SIZE  = 16  if SMOKE_TEST else 128
MAX_SAMPLES = 600 if SMOKE_TEST else None

_manifest = PROJECT_ROOT / "dataset" / "metadata" / "train_manifest.csv"
if not _manifest.exists() or _manifest.stat().st_size < 1000:
    env.download_manifest(_manifest)
if not _manifest.exists() or _manifest.stat().st_size < 1000:
    env.rebuild_manifest_from_kaggle(_manifest)
if _manifest.exists(): cfg.dataset.metadata_paths = [str(_manifest)]

_split_name = "split_indices_smoke.json" if SMOKE_TEST else "split_indices.json"
_split_path = PROJECT_ROOT / "dataset" / "metadata" / _split_name
if not _split_path.exists() and not SMOKE_TEST and env.hf_token:
    try:
        from huggingface_hub import hf_hub_download
        import shutil
        _dl = hf_hub_download(repo_id=env.hf_manifest_repo, filename="split_indices.json", repo_type="model", token=env.hf_token)
        shutil.copy2(_dl, _split_path)
    except Exception as e: print(f"Warning: could not download split_indices from HF: {e}")

train_loader, val_loader, test_loader = create_split_dataloaders(
    root_dir=str(PROJECT_ROOT), metadata_paths=[str(_manifest)],
    batch_size=BATCH_SIZE, num_workers=0 if SMOKE_TEST else 4, size=IMAGE_SIZE,
    val_split=0.10, test_split=0.10, seed=SEED, use_weighted_sampler=False,
    split_index_path=str(_split_path), max_samples=MAX_SAMPLES
)

OUT = PROJECT_ROOT / "paper" / "result" / ("verify" if SMOKE_TEST else "full_scale") / "paper_evals"
OUT.mkdir(parents=True, exist_ok=True)
print(f"Test Set: {len(test_loader.dataset)} samples")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 3: Load Checkpoint
# ═══════════════════════════════════════════════════════════
from src.model import build_mfft, count_parameters

def load_variant(v):
    m = build_mfft(v)
    ckpt_dir = PROJECT_ROOT / "model" / "checkpoints" / f"{v}_model"
    # Attempt HF download
    env.download_latest_checkpoint(ckpt_dir, f"{v}_model")
    # Local search
    for ck in [ckpt_dir / f"best_mfft_{v}.pt", ckpt_dir / "best.pt"]:
        if ck.exists():
            try:
                m.load_state_dict(torch.load(ck, map_location="cpu", weights_only=True))
                print(f"{v}: Loaded {ck.name}")
                return m.to(device).eval(), str(ck)
            except: pass
    print(f"{v}: NO CHECKPOINT. Using random init.")
    return m.to(device).eval(), None

model, ckpt_path = load_variant(VARIANT)
print(f"Params: {count_parameters(model):,}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 4: Collect Logits
# ═══════════════════════════════════════════════════════════
@torch.no_grad()
def collect(loader, net):
    logits, labels = [], []
    for x, y in tqdm(loader):
        logits.append(net(x.to(device)).cpu()); labels.append(y)
    return torch.cat(logits), torch.cat(labels)

print("Collecting val logits (for temp scaling)...")
val_logits, val_labels = collect(val_loader, model)
print("Collecting test logits...")
test_logits, test_labels = collect(test_loader, model)

y_true = test_labels.numpy()
y_score = F.softmax(test_logits, dim=-1)[:, 1].numpy()
y_pred = (y_score >= 0.5).astype(int)
acc = (y_pred == y_true).mean() * 100
print(f"Test Acc: {acc:.2f}%")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 5: Confidence Intervals (Bootstrap)
# ═══════════════════════════════════════════════════════════
from src.stats import bootstrap_ci
rows = []
for metric, arg in [('accuracy', y_pred), ('precision', y_pred), ('recall', y_pred), ('f1', y_pred), ('auc', y_score)]:
    ci = bootstrap_ci(y_true, arg, metric=metric, n_resamples=1000, seed=SEED)
    rows.append({'metric': metric, 'point': round(ci['point'], 4), 'ci_lower': round(ci['lower'], 4), 'ci_upper': round(ci['upper'], 4)})
    print(f"{metric:>10}: {ci['point']:.4f}  [{ci['lower']:.4f}, {ci['upper']:.4f}]")

pd.DataFrame(rows).to_csv(OUT / f"{VARIANT}_metrics_ci.csv", index=False)

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 6: Temperature Scaling & Calibration
# ═══════════════════════════════════════════════════════════
from src.calibration import TemperatureScaler, expected_calibration_error, brier_score
pre_ece, pre_brier = expected_calibration_error(y_true, y_score), brier_score(y_true, y_score)

scaler = TemperatureScaler()
T = scaler.fit(val_logits, val_labels)
cal_score = scaler.calibrate(test_logits)[:, 1].numpy()
post_ece, post_brier = expected_calibration_error(y_true, cal_score), brier_score(y_true, cal_score)

print(f"Temperature: {T:.3f}")
print(f"ECE   pre={pre_ece:.4f}  post={post_ece:.4f}")
print(f"Brier pre={pre_brier:.4f}  post={post_brier:.4f}")

pd.DataFrame([{'temperature': round(T, 4), 'ece_pre': round(pre_ece, 4), 'ece_post': round(post_ece, 4),
               'brier_pre': round(pre_brier, 4), 'brier_post': round(post_brier, 4)}]).to_csv(OUT / f"{VARIANT}_calibration.csv", index=False)

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 7: McNemar Test (vs Tiny)
# ═══════════════════════════════════════════════════════════
from src.stats import mcnemar_test
other, other_ck = load_variant('tiny')
other_logits, _ = collect(test_loader, other)
other_pred = other_logits.argmax(dim=-1).numpy()

res = mcnemar_test(y_true, y_pred, other_pred)
print(res)
pd.DataFrame([{'model_a': VARIANT, 'model_b': 'tiny', **res}]).to_csv(OUT / "mcnemar.csv", index=False)

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 8: Robustness Suite (JPEG, Blur, Noise)
# ═══════════════════════════════════════════════════════════
from src.robustness import RobustnessEvaluator
evaluator = RobustnessEvaluator(model, device, size=IMAGE_SIZE, batch_size=BATCH_SIZE)
subset = test_dataset.samples[:60] if SMOKE_TEST else test_dataset.samples[:1000]
results = evaluator.evaluate(subset)

for t, q, acc in results:
    print(f"{t} @ {q}: {acc:.2f}%")
pd.DataFrame(results, columns=['transform', 'quality', 'accuracy']).to_csv(OUT / "robustness.csv", index=False)

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 9: LOGO (Prepare Manifests)
# ═══════════════════════════════════════════════════════════
from src.logo_eval import create_logo_manifests
logo_dir = PROJECT_ROOT / "dataset" / "metadata" / "logo"
logo_dir.mkdir(parents=True, exist_ok=True)
create_logo_manifests(cfg.dataset.metadata_paths[0], logo_dir)

print("\nUploading Paper Evals to HF...")
env.upload_results_dir(OUT, "paper_evals")

print("\n✅ Paper Evals complete. Proceed to 07_logo_eval.ipynb")